# 10 — Capstone: End-to-End ML Project

## Business Problem

**Company:** A B2B SaaS firm with 7,000 customers across multiple industries.

**Problem:** Customer churn is at 18% annually, costing ~$4M/year. The CS team can proactively intervene with at-risk accounts, but they only have capacity to contact 200 customers per month. They need a model that:
1. Predicts which customers will churn in the next 90 days.
2. Ranks them so the 200 highest-risk accounts are prioritized.
3. Explains *why* a customer is at risk (interpretability requirement).
4. Segments the customer base to design targeted retention offers.

**Your deliverable:** A complete ML pipeline from raw data to a scored, segmented, explainable output.

---

**Rules:**
- `pandas`, `numpy`, `scikit-learn` only.
- Every function must have assertions you verify before moving on.
- You must write a markdown cell after each section interpreting results in business terms.
- Final output must be a single DataFrame `final_output` that the CS team could act on.

**Reference:** [sklearn docs](https://scikit-learn.org/stable/)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, precision_recall_curve
)
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

# Telecom churn dataset — 20 features, ~7k customers, binary churn target
churn_raw = fetch_openml(name='telco-customer-churn', version=1, as_frame=True, parser='auto').frame

# Fallback: if above fails, use IBM Telco from sklearn alternative
if churn_raw is None or len(churn_raw) < 100:
    from sklearn.datasets import make_classification
    X_fallback, y_fallback = make_classification(
        n_samples=7000, n_features=20, n_informative=10,
        n_redundant=4, random_state=42
    )

print(churn_raw.shape)
churn_raw.head()

---
## Phase 1 — Exploratory Data Analysis & Data Quality

**Before building any model, understand the data.**

### 1.1 — Data Quality Assessment

Write `full_eda_report(df)` that returns a dict with:
- `shape`: (rows, cols)
- `target_rate`: churn rate (proportion of positive class)
- `null_summary`: DataFrame — columns with nulls, count, pct
- `dtype_summary`: count of numeric vs categorical columns
- `duplicate_rows`: count of exact duplicate rows
- `numeric_stats`: `.describe()` on numeric columns

This function must work on any DataFrame.

In [ ]:
def full_eda_report(df: pd.DataFrame, target_col: str) -> dict:
    """
    Comprehensive EDA report.
    """
    # YOUR CODE HERE
    pass

# Identify the target column — inspect churn_raw.columns and dtypes first
# Then call:
# eda = full_eda_report(churn_raw, target_col='...')

In [ ]:
# --- ASSERTIONS ---
# eda = full_eda_report(churn_raw, target_col='Churn')  # adjust column name
# assert set(eda.keys()) == {'shape', 'target_rate', 'null_summary', 'dtype_summary', 'duplicate_rows', 'numeric_stats'}
# assert 0 < eda['target_rate'] < 1
# assert eda['shape'] == churn_raw.shape
# print("✓ Phase 1.1 passed")
# print(f"Churn rate: {eda['target_rate']:.2%}")

### 1.2 — Feature Distributions & Univariate Churn Rates

For each **categorical** feature, compute the churn rate per category.
For each **numeric** feature, compute mean value for churned vs non-churned customers.

Return:
- `categorical_churn_rates`: dict of {feature: DataFrame(category, count, churn_rate)} sorted by churn_rate desc.
- `numeric_churn_diff`: DataFrame with columns `feature`, `mean_churned`, `mean_retained`, `abs_diff`, `pct_diff`. Sorted by abs_diff desc.

In [ ]:
def univariate_churn_analysis(df: pd.DataFrame, target_col: str):
    """
    Returns (categorical_churn_rates, numeric_churn_diff)
    """
    # YOUR CODE HERE
    pass

# categorical_churn_rates, numeric_churn_diff = univariate_churn_analysis(churn_raw, 'Churn')

In [ ]:
# --- ASSERTIONS ---
# assert isinstance(categorical_churn_rates, dict)
# assert list(numeric_churn_diff.columns) == ['feature', 'mean_churned', 'mean_retained', 'abs_diff', 'pct_diff']
# assert numeric_churn_diff['abs_diff'].is_monotonic_decreasing
# print("✓ Phase 1.2 passed")
# print("Top predictors by raw mean difference:")
# print(numeric_churn_diff.head())

**Business interpretation:** *(Which features show the largest separation between churned and retained customers? What does this suggest about churn drivers?)*

---
## Phase 2 — Feature Engineering

### 2.1 — Domain-Driven Feature Creation

Based on your EDA findings, create business-relevant features.

Implement `ChurnFeatureEngineer` — a custom sklearn transformer that adds:
1. `tenure_band`: bucket `tenure` into `'New'` (0–12mo), `'Developing'` (13–24mo), `'Mature'` (25–48mo), `'Loyal'` (49+mo).
2. `monthly_charge_per_service`: `MonthlyCharges / (number of active services)` — efficiency metric.
3. `charge_vs_tenure_interaction`: `MonthlyCharges * log(tenure + 1)` — high charges early = high risk.
4. `has_support_contract`: 1 if `TechSupport == 'Yes'` OR `OnlineBackup == 'Yes'`, else 0.
5. Any 2 additional features you believe are predictive based on your EDA. Document your reasoning.

In [ ]:
class ChurnFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Domain-driven feature engineering for churn prediction.
    Stateless — fit() does nothing.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        # YOUR CODE HERE
        pass

In [ ]:
# --- ASSERTIONS ---
# cfe = ChurnFeatureEngineer()
# test_out = cfe.fit_transform(churn_raw.drop('Churn', axis=1))
# for col in ['tenure_band', 'monthly_charge_per_service', 'charge_vs_tenure_interaction', 'has_support_contract']:
#     assert col in test_out.columns, f"Missing: {col}"
# assert test_out.shape[0] == len(churn_raw)
# print("✓ Phase 2.1 passed")

**Feature rationale:** *(Explain why each custom feature might predict churn — this is what interviewers look for)*

### 2.2 — Full Preprocessing Pipeline

Build `build_preprocessing_pipeline(numeric_cols, categorical_cols)` that returns a `ColumnTransformer`:
- Numeric: median imputation → StandardScaler
- Categorical: most-frequent imputation → OneHotEncoder(handle_unknown='ignore', drop='first')

Then build `full_preprocess_pipeline`: `ChurnFeatureEngineer` → `ColumnTransformer`.

In [ ]:
def build_preprocessing_pipeline(numeric_cols, categorical_cols):
    """
    Returns a ColumnTransformer.
    """
    # YOUR CODE HERE
    pass

# YOUR CODE: identify numeric and categorical columns after feature engineering
# Build full_preprocess_pipeline

---
## Phase 3 — Model Development

### 3.1 — Train/Test Split

Split data: 80/20, stratified on target, `random_state=42`.
Print class balance for both splits — verify stratification held.

In [ ]:
# YOUR CODE HERE
# X_train, X_test, y_train, y_test = ...

### 3.2 — Baseline Model

**Always start with a baseline.** A model that predicts the majority class gives you the floor to beat.

Implement `DummyMajorityClassifier` that always predicts the majority class. Evaluate it with your `evaluate_classifier` function. This is your performance floor.

In [ ]:
class DummyMajorityClassifier(BaseEstimator):
    """
    Predicts majority class always.
    """
    def fit(self, X, y):
        # YOUR CODE HERE
        pass

    def predict(self, X):
        # YOUR CODE HERE
        pass

    def predict_proba(self, X):
        # YOUR CODE HERE
        pass

### 3.3 — Model Comparison

Train and evaluate **3 models** using 5-fold stratified CV:
1. `LogisticRegression(max_iter=1000, C=1.0)`
2. `RandomForestClassifier(n_estimators=100, random_state=42)`
3. `GradientBoostingClassifier(n_estimators=100, random_state=42)`

Each inside a full pipeline (preprocessing + model).

Evaluate on: `roc_auc`, `f1`, `precision`, `recall`.

Return `model_cv_results`: DataFrame with `model`, `roc_auc_mean`, `roc_auc_std`, `f1_mean`, `f1_std`, `precision_mean`, `recall_mean`.

In [ ]:
def compare_models_cv(X_train, y_train, preprocessor):
    """
    Returns model_cv_results DataFrame.
    """
    # YOUR CODE HERE
    pass

# model_cv_results = compare_models_cv(X_train, y_train, full_preprocess_pipeline)

In [ ]:
# --- ASSERTIONS ---
# assert len(model_cv_results) == 3
# assert (model_cv_results['roc_auc_mean'] > 0.6).all(), "All models must beat 0.6 AUC"
# print("✓ Phase 3.3 passed")
# print(model_cv_results)

**Model selection rationale:** *(Which model do you choose and why? Consider: AUC, F1, interpretability, training time)*

### 3.4 — Hyperparameter Tuning

Tune your chosen model with `RandomizedSearchCV` (n_iter=20, 5-fold CV, scoring='roc_auc').

Define a meaningful parameter distribution for your chosen model.

Return `best_model_pipeline`: the best fitted full pipeline.

In [ ]:
def tune_best_model(X_train, y_train, preprocessor):
    """
    Returns (best_model_pipeline, best_params, best_cv_auc)
    """
    # YOUR CODE HERE
    pass

# best_model_pipeline, best_params, best_cv_auc = tune_best_model(X_train, y_train, full_preprocess_pipeline)

---
## Phase 4 — Business-Aligned Evaluation

### 4.1 — Threshold Optimization for Business Constraint

**Constraint:** CS team can contact 200 customers/month. You must identify the decision threshold that flags exactly (or as close as possible to) 200 customers in the test set as high-risk.

1. Get predicted probabilities from `best_model_pipeline` on test set.
2. Find `capacity_threshold`: the probability cutoff that results in ≤ 200 predicted positives.
3. At this threshold, compute: `precision` (what fraction are actually at risk?), `recall` (what fraction of actual churners do we catch?), `expected_saved_customers` = `true_positives` (assuming 100% intervention success).
4. Return `business_eval`: dict with `capacity_threshold`, `n_flagged`, `precision`, `recall`, `expected_saved_customers`.

In [ ]:
def evaluate_business_constraint(model, X_test, y_test, capacity=200):
    """
    Returns business_eval dict aligned to CS team capacity constraint.
    """
    # YOUR CODE HERE
    pass

# business_eval = evaluate_business_constraint(best_model_pipeline, X_test, y_test, capacity=200)

In [ ]:
# --- ASSERTIONS ---
# assert set(business_eval.keys()) == {'capacity_threshold', 'n_flagged', 'precision', 'recall', 'expected_saved_customers'}
# assert business_eval['n_flagged'] <= 200
# assert 0 < business_eval['precision'] <= 1
# print("✓ Phase 4.1 passed")
# print(business_eval)

**Business interpretation:** *(If the model saves X customers from churning, and average contract value is $Y, what is the ROI? Make reasonable assumptions and calculate.)*

### 4.2 — Model Interpretability

The CS team needs to know *why* a customer is at risk, not just that they are.

1. Extract feature importances from your best model (or coefficients if logistic regression).
2. Build `top10_features`: top 10 most important features with their importance scores.
3. For the **top 20 highest-risk customers** in the test set, build `risk_explanations`: a DataFrame showing their feature values for the top 10 features + their churn probability. This is what the CS team would see.

In [ ]:
def build_risk_explanations(model, X_test, y_test, top_n_features=10, top_n_customers=20):
    """
    Returns (top10_features DataFrame, risk_explanations DataFrame)
    """
    # YOUR CODE HERE
    pass

# top10_features, risk_explanations = build_risk_explanations(best_model_pipeline, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
# assert len(top10_features) == 10
# assert 'churn_probability' in risk_explanations.columns
# assert len(risk_explanations) == 20
# assert risk_explanations['churn_probability'].is_monotonic_decreasing
# print("✓ Phase 4.2 passed")

---
## Phase 5 — Customer Segmentation

### 5.1 — Cluster the At-Risk Population

Customers flagged as high-risk should receive **different** retention offers based on their profile.

1. Take all customers in the test set with `churn_probability >= capacity_threshold`.
2. Apply KMeans (k=3) on their **original numeric features** (standardized).
3. Profile each segment: mean of each numeric feature per cluster.
4. Assign business-meaningful names to each cluster based on their profile (e.g., `'Price Sensitive'`, `'Low Engagement'`, `'Service Issues'`).
5. Return `segment_profiles`: DataFrame with cluster profiles + your assigned name.

In [ ]:
def segment_at_risk_customers(X_test, churn_proba, threshold, k=3):
    """
    Returns (X_at_risk_segmented, segment_profiles)
    X_at_risk_segmented: at-risk customers with cluster labels and churn_probability
    segment_profiles: cluster means with segment name
    """
    # YOUR CODE HERE
    pass

# churn_proba = best_model_pipeline.predict_proba(X_test)[:, 1]
# X_at_risk_segmented, segment_profiles = segment_at_risk_customers(
#     X_test, churn_proba, business_eval['capacity_threshold'], k=3
# )

In [ ]:
# --- ASSERTIONS ---
# assert 'cluster' in X_at_risk_segmented.columns
# assert 'churn_probability' in X_at_risk_segmented.columns
# assert 'segment_name' in segment_profiles.columns
# assert len(segment_profiles) == 3
# print("✓ Phase 5.1 passed")
# print(segment_profiles)

**Retention offer recommendations:**
- Segment A (*Price Sensitive*): *...*
- Segment B (*Low Engagement*): *...*
- Segment C (*Service Issues*): *...*

---
## Phase 6 — Final Output

### 6.1 — Build the CS Team Action File

Combine everything into a single actionable DataFrame `final_output` that the Customer Success team can open in Excel.

Each row = one at-risk customer. Columns:
- `customer_id` (or index)
- `churn_probability` (sorted descending)
- `risk_tier`: `'Critical'` (top 25%), `'High'` (25–50%), `'Medium'` (50–75%), `'Monitor'` (bottom 25%)
- `segment_name`: from Phase 5
- `suggested_action`: based on segment (`'Offer discount'`, `'Assign CSM'`, `'Product training'`, etc.)
- Top 3 feature values that most influence their score (one column per feature)
- `actual_churned`: ground truth (for model validation after the fact)

In [ ]:
def build_final_output(X_at_risk_segmented, segment_profiles, top10_features):
    """
    Returns final_output DataFrame — the CS team action file.
    """
    # YOUR CODE HERE
    pass

# final_output = build_final_output(X_at_risk_segmented, segment_profiles, top10_features)

In [ ]:
# --- ASSERTIONS ---
# required_cols = ['churn_probability', 'risk_tier', 'segment_name', 'suggested_action', 'actual_churned']
# for col in required_cols:
#     assert col in final_output.columns, f"Missing: {col}"
# assert final_output['churn_probability'].is_monotonic_decreasing
# assert set(final_output['risk_tier'].unique()) == {'Critical', 'High', 'Medium', 'Monitor'}
# assert final_output['churn_probability'].between(0, 1).all()
# print("✓ Phase 6.1 passed — Final output ready")
# print(final_output.head(10))

---
## Phase 7 — Project Summary

### 7.1 — Executive Summary Table

Build `project_summary`: a single DataFrame summarizing every key result in this project.

Columns: `phase`, `finding`, `metric_value`, `business_implication`.

Must include at least 8 rows covering: churn rate, baseline model, best model AUC, F1, capacity threshold, expected saves, top feature, number of segments.

In [ ]:
def build_project_summary():
    """
    Returns project_summary DataFrame — executive summary.
    """
    # YOUR CODE HERE
    pass

# project_summary = build_project_summary()

In [ ]:
# --- ASSERTIONS ---
# assert list(project_summary.columns) == ['phase', 'finding', 'metric_value', 'business_implication']
# assert len(project_summary) >= 8
# print("✓ Phase 7.1 passed — Capstone complete")
# print(project_summary.to_string())

---
## Self-Review Checklist

Before considering this capstone complete, verify:

- [ ] All assert blocks pass without errors
- [ ] No data leakage: all preprocessing fit only on `X_train`
- [ ] Every function has a docstring
- [ ] Every section has a business interpretation in markdown
- [ ] `final_output` is sorted by `churn_probability` descending
- [ ] Model choice is justified in writing
- [ ] Threshold choice is justified by the business constraint, not just a technical metric
- [ ] Segment names are interpretable to a non-technical stakeholder
- [ ] Executive summary has at least 8 rows

**This checklist is what a senior data scientist would use to review your work in a BCG X interview.**